# Python Logging: LogRecord Flow, Levels, and Filters

This notebook provides a detailed explanation of how Python's `logging` module works, specifically how a `logging.LogRecord` moves from the log site to being emitted by a handler, and how levels and filters affect this flow.

## Core Components

Before diving into the flow, let's understand the key components:

1. **Logger** (`logging.Logger`) - The entry point for logging. Applications call methods like `logger.info()`, `logger.warning()`, etc. Loggers form a hierarchy based on their names (e.g., `foo.bar` is a child of `foo`).

2. **LogRecord** (`logging.LogRecord`) - A data object containing all the information about a logging event: message, level, timestamp, source location, etc.

3. **Handler** (`logging.Handler`) - Responsible for dispatching log records to a destination (console, file, network, etc.). A logger can have multiple handlers.

4. **Filter** (`logging.Filter`) - A mechanism to provide fine-grained control over which log records pass through a logger or handler.

5. **Formatter** (`logging.Formatter`) - Converts a LogRecord into a string (or other format) for output.

6. **Level** - A numeric severity indicator. Standard levels are DEBUG(10), INFO(20), WARNING(30), ERROR(40), CRITICAL(50).

## The Complete LogRecord Flow

Here's the complete journey of a log record from creation to output:

```mermaid
flowchart TD
    A["1. LOG CALL<br/>logger.info('message')"] --> B["2. EARLY LEVEL CHECK<br/>logger.isEnabledFor(level)?"]
    
    B --> B1["Call getEffectiveLevel()"]
    
    subgraph EffLevel ["getEffectiveLevel() - walks up hierarchy"]
        direction TB
        EL1["Start at current logger"] --> EL2{"logger.level != NOTSET?"}
        EL2 -->|"✅ Yes"| EL3["Return logger.level"]
        EL2 -->|"❌ No (NOTSET)"| EL4{"Has parent?"}
        EL4 -->|"✅ Yes"| EL5["Move to parent"] --> EL2
        EL4 -->|"❌ No (at root)"| EL6["Return NOTSET (0)"]
    end
    
    B1 --> EffLevel
    EL3 --> B2{"record_level >= effective_level?"}
    EL6 --> B2
    
    B2 -->|❌ No| STOP1[/"STOP<br/>No LogRecord created"/]
    B2 -->|✅ Yes| C["3. CREATE LogRecord<br/><i>Captures: msg, args, level,<br/>pathname, lineno, func, exc_info, etc.</i>"]
    
    C --> D{"4. LOGGER FILTER CHECK<br/>Do all logger.filters pass?"}
    
    D -->|"Any returns False"| STOP2[/"STOP<br/>Record dropped"/]
    D -->|"All pass"| E["5. CALL HANDLERS<br/>For each handler in logger.handlers"]
    
    E --> F{"5a. HANDLER LEVEL CHECK<br/>record.levelno >= handler.level?"}
    
    F -->|❌ No| SKIP["Skip this handler"]
    F -->|✅ Yes| G{"5b. HANDLER FILTER CHECK<br/>Do all handler.filters pass?"}
    
    G -->|"Any returns False"| SKIP
    G -->|"All pass"| H["5c. EMIT<br/>handler.emit(record)<br/><i>Format & write to destination</i>"]
    
    H --> I{More handlers?}
    SKIP --> I
    I -->|✅ Yes| F
    I -->|❌ No| J{"6. PROPAGATE?<br/>logger.propagate == True?"}
    
    J -->|❌ No| DONE[/"DONE"/]
    J -->|"✅ Yes → move to parent"| K{"Has parent logger?"}
    
    K -->|❌ No| DONE
    K -->|"✅ Yes<br/><i>Parent filters NOT checked!</i>"| E
    
    style STOP1 fill:#450a0a
    style STOP2 fill:#450a0a
    style SKIP fill:#422006
    style DONE fill:#052e16
    style H fill:#052e16
    style EffLevel fill:#1e1b4b,stroke:#6366f1
    style EL3 fill:#052e16
    style EL6 fill:#052e16
```

## Step-by-Step Breakdown

### Step 1: The Log Call

When you call a logging method, you're calling a method on a `Logger` instance:

In [ ]:
import logging

# Get a logger (creates it if it doesn't exist)
logger = logging.getLogger("myapp.module")

# These all eventually call logger._log() with the appropriate level
logger.debug("Debug message")  # level = DEBUG (10)
logger.info("Info message")  # level = INFO (20)
logger.warning("Warning message")  # level = WARNING (30)
logger.error("Error message")  # level = ERROR (40)
logger.critical("Critical!")  # level = CRITICAL (50)

# You can also use the generic log() method
logger.log(logging.INFO, "Same as logger.info()")

### Step 2: The Early Level Check (isEnabledFor)

Before creating a `LogRecord`, the logger checks if the message would even be processed. This is an optimization to avoid the overhead of creating `LogRecord` objects that would just be discarded.

The check uses `logger.isEnabledFor(level)`, which compares against the logger's **effective level**.

In [ ]:
# The effective level is determined by walking up the hierarchy
# until a logger with an explicitly set level is found

root = logging.getLogger()  # Root logger
parent = logging.getLogger("myapp")
child = logging.getLogger("myapp.module")

# Initially, no level is set (NOTSET = 0)
print(f"child.level = {child.level}")  # 0 (NOTSET)
print(f"parent.level = {parent.level}")  # 0 (NOTSET)
print(f"root.level = {root.level}")  # WARNING (30) by default

# Effective level walks up until it finds a non-NOTSET level
print(
    f"child.getEffectiveLevel() = {child.getEffectiveLevel()}"
)  # 30 (inherited from root)

In [ ]:
# If we set the parent's level, children inherit it
parent.setLevel(logging.DEBUG)
print(f"parent.level = {parent.level}")  # 10 (DEBUG)
print(
    f"child.getEffectiveLevel() = {child.getEffectiveLevel()}"
)  # 10 (now inherited from parent)

# Setting the child's level overrides inheritance
child.setLevel(logging.ERROR)
print(
    f"child.getEffectiveLevel() = {child.getEffectiveLevel()}"
)  # 40 (ERROR, explicitly set)

**Key Point**: The early level check happens *before* the `LogRecord` is created. If the effective level is higher than the log call's level, no `LogRecord` is ever created. This is important for performance when you have many debug statements that won't be output.

### Step 3: LogRecord Creation

If the early level check passes, a `LogRecord` is created. This object captures everything about the logging event:

In [ ]:
# A LogRecord has many attributes. Here are the key ones:
record = logging.LogRecord(
    name="myapp.module",  # Logger name
    level=logging.INFO,  # Numeric level (20)
    pathname="/path/to/file.py",
    lineno=42,
    msg="User %s logged in",  # The format string
    args=("alice",),  # Arguments for % formatting
    exc_info=None,  # Exception info tuple, if any
)

# Some important attributes:
print(f"record.name = {record.name}")  # 'myapp.module'
print(f"record.levelno = {record.levelno}")  # 20
print(f"record.levelname = {record.levelname}")  # 'INFO'
print(f"record.getMessage() = {record.getMessage()}")  # 'User alice logged in'
print(f"record.created = {record.created}")  # Unix timestamp

### Step 4: Logger Filter Check

After the `LogRecord` is created, it must pass through the logger's filters. Filters are checked by calling the logger's `filter()` method, which iterates through all attached filters.

In [ ]:
# Filters can be:
# 1. An instance of logging.Filter (or subclass)
# 2. Any callable that takes a LogRecord and returns a truthy/falsy value

# Example 1: Built-in Filter class filters by logger name prefix
name_filter = logging.Filter(name="myapp.database")
# This filter only allows records from loggers named "myapp.database"
# or children like "myapp.database.queries"


# Example 2: Custom filter class
class LevelRangeFilter(logging.Filter):
    """Only allow records within a level range."""

    def __init__(self, min_level=logging.DEBUG, max_level=logging.CRITICAL):
        super().__init__()
        self.min_level = min_level
        self.max_level = max_level

    def filter(self, record: logging.LogRecord) -> bool:
        return self.min_level <= record.levelno <= self.max_level


# Example 3: Simple callable (function)
def no_passwords(record: logging.LogRecord) -> bool:
    """Filter out any log messages containing 'password'."""
    return "password" not in record.getMessage().lower()

In [ ]:
# Adding filters to a logger
logger = logging.getLogger("myapp")
logger.addFilter(no_passwords)
logger.addFilter(LevelRangeFilter(logging.INFO, logging.ERROR))

# Now when logger.handle(record) is called, both filters must pass
# The filtering logic is in Filterer.filter() (base class of Logger):
#
#   def filter(self, record):
#       for f in self.filters:
#           if hasattr(f, 'filter'):
#               result = f.filter(record)
#           else:
#               result = f(record)  # callable
#           if not result:
#               return False
#       return True

### Step 5: Handler Processing

If the record passes the logger's filters, it's passed to each handler attached to that logger. Each handler performs its own level check and filter check before emitting the record.

In [ ]:
# Setting up handlers with different levels and filters

import sys

# Create handlers
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)  # Only INFO and above

file_handler = logging.FileHandler("/tmp/app.log")
file_handler.setLevel(logging.DEBUG)  # Everything DEBUG and above

error_handler = logging.FileHandler("/tmp/errors.log")
error_handler.setLevel(logging.ERROR)  # Only ERROR and CRITICAL

# Each handler can also have its own filters
console_handler.addFilter(no_passwords)  # Additional filtering on console

# Attach to logger
logger = logging.getLogger("myapp")
logger.addHandler(console_handler)
logger.addHandler(file_handler)
logger.addHandler(error_handler)

The handler's `handle()` method performs the checks:

```python
# Simplified version of Handler.handle()
def handle(self, record):
    # Step 5a: Handler level check
    if record.levelno < self.level:
        return  # Skip this handler
    
    # Step 5b: Handler filter check  
    if not self.filter(record):  # Same logic as Logger.filter()
        return  # Skip this handler
    
    # Step 5c: Emit the record
    self.emit(record)
```

**Key Difference from Logger Level Check**: The handler level check compares `record.levelno >= handler.level` directly. Unlike the logger's *effective* level, a handler's level is not inherited from anywhere—it's explicitly set or defaults to `NOTSET` (0), which means "accept everything."

### Step 6: Propagation

After processing all handlers on the current logger, the `callHandlers()` method checks if the record should propagate to parent loggers.

**Critical Detail**: When a record propagates, parent loggers' **filters are NOT checked**—only their **handlers** are invoked (with the handlers' own level and filter checks).

In [ ]:
# Simplified version of Logger.callHandlers()
def callHandlers(self, record):
    c = self
    found = 0
    while c:
        for handler in c.handlers:
            found += 1
            if record.levelno >= handler.level:
                handler.handle(record)  # Handler does its own filter check

        # Move to parent if propagate is True
        if not c.propagate:
            break
        c = c.parent

    # If no handlers found anywhere, use lastResort handler
    if found == 0:
        if logging.lastResort:
            logging.lastResort.handle(record)

## Summary: Where Levels and Filters Apply

| Check | Where | What's Checked | Effect if Fails |
|-------|-------|----------------|-----------------|
| **Early level check** | Logger (before LogRecord) | `record_level >= logger.getEffectiveLevel()` | No LogRecord created |
| **Logger filter check** | Logger (after LogRecord) | All `logger.filters` must pass | Record dropped entirely |
| **Handler level check** | Each Handler | `record.levelno >= handler.level` | Handler skipped |
| **Handler filter check** | Each Handler | All `handler.filters` must pass | Handler skipped |

### Important Implications:

1. **Logger level is a gate**: If a logger's effective level is INFO, DEBUG records never create LogRecords—they're discarded immediately for performance.

2. **Logger filters are global for that logger**: If a record fails a logger's filter, NO handlers see it (including parent handlers via propagation).

3. **Handler level allows per-destination control**: You can have one handler for DEBUG+ (file) and another for ERROR+ (console) on the same logger.

4. **Handler filters are local**: A record can pass one handler's filters but fail another's on the same logger.

5. **Propagation bypasses parent logger filters**: Only the originating logger's filters are checked. Parent loggers' filters are ignored—only their handlers' level/filter checks apply.

In [ ]:
import logging
import sys

# Reset logging for clean example
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Setup: root logger with console handler at WARNING
root = logging.getLogger()
root.setLevel(logging.DEBUG)  # Root accepts everything

console = logging.StreamHandler(sys.stderr)
console.setLevel(logging.WARNING)  # But console only shows WARNING+
console.setFormatter(
    logging.Formatter("%(name)s - %(levelname)s - %(message)s")
)
root.addHandler(console)

# Child logger with no explicit level (inherits DEBUG from root)
# and no handlers (will propagate to root's handler)
app_logger = logging.getLogger("myapp")

In [ ]:
# Trace 1: app_logger.debug("Hello")
#
# 1. Log call: app_logger.debug("Hello")
# 2. Early level check: is DEBUG(10) >= app_logger.getEffectiveLevel()?
#    - app_logger.level = NOTSET(0), so check parent
#    - root.level = DEBUG(10)
#    - Effective level = 10, so 10 >= 10 ✓ PASS
# 3. Create LogRecord(level=DEBUG, msg="Hello", ...)
# 4. Logger filter check: app_logger has no filters ✓ PASS
# 5. Call handlers: app_logger has no handlers (empty list)
# 6. Propagate? app_logger.propagate = True, so move to root
# 5. (at root) Call handlers:
#    - console handler level check: 10 >= 30? ✗ FAIL
#    - Handler skipped!
# Result: Nothing printed

app_logger.debug("Hello")  # Silent - handler level blocks it

In [ ]:
# Trace 2: app_logger.warning("World")
#
# 1. Log call: app_logger.warning("World")
# 2. Early level check: is WARNING(30) >= effective_level(10)? ✓ PASS
# 3. Create LogRecord(level=WARNING, msg="World", ...)
# 4. Logger filter check: no filters ✓ PASS
# 5. Call handlers: app_logger has no handlers
# 6. Propagate to root
# 5. (at root) Call handlers:
#    - console handler level check: 30 >= 30? ✓ PASS
#    - console handler filter check: no filters ✓ PASS
#    - emit! → prints to stderr
# Result: "myapp - WARNING - World"

app_logger.warning("World")  # Prints!

## The Special Role of NOTSET (0)

`NOTSET` (value 0) has different meanings for loggers and handlers:

### For Loggers:
- `NOTSET` means "inherit from parent"
- `getEffectiveLevel()` walks up the hierarchy until it finds a non-NOTSET level
- If even the root logger is NOTSET, the effective level is NOTSET (0), which allows everything

### For Handlers:
- `NOTSET` means "accept all records" (no level filtering)
- Handlers don't inherit levels—they just use their own level directly
- Default handler level is NOTSET

In [ ]:
# Demonstrating NOTSET behavior

logger = logging.getLogger("test.notset")

# Logger with NOTSET inherits from parent chain
logger.setLevel(logging.NOTSET)
print(f"logger.level = {logger.level}")  # 0 (NOTSET)
print(f"logger.getEffectiveLevel() = {logger.getEffectiveLevel()}")  # Inherited

# Handler with NOTSET accepts everything
handler = logging.StreamHandler()
print(f"handler.level = {handler.level}")  # 0 (NOTSET) - accepts all levels

## Common Patterns and Gotchas

### Pattern 1: Central Handler Configuration

A common pattern is to attach handlers only to the root logger and let propagation do the work:

In [ ]:
# All handlers on root, child loggers just log
logging.basicConfig(
    level=logging.DEBUG,
    handlers=[
        logging.StreamHandler(),  # Console
        logging.FileHandler("/tmp/app.log"),  # File
    ],
)

# Child loggers have no handlers—they propagate to root
db_logger = logging.getLogger("myapp.database")
api_logger = logging.getLogger("myapp.api")

# Both use root's handlers via propagation
db_logger.info("Query executed")  # Goes to console and file
api_logger.warning("Rate limited")  # Goes to console and file

### Gotcha 1: Duplicate Messages

If you add handlers to both a child logger AND the root logger, and propagation is enabled, you'll get duplicate messages:

In [ ]:
# Problem: Duplicate messages
child = logging.getLogger("myapp.child")
child.addHandler(logging.StreamHandler())  # Handler on child
# Root also has a handler from basicConfig above

child.warning("Oops")
# Prints TWICE:
# 1. From child's handler
# 2. From root's handler (via propagation)

# Solution 1: Set propagate=False on child
child.propagate = False

# Solution 2: Don't add handlers to child loggers (preferred)

### Gotcha 2: Parent Filters Don't Apply to Propagated Records

This is a common source of confusion. If you add a filter to a parent logger, it won't affect records that propagate from child loggers:

In [ ]:
# Filter on root logger
def block_all(record):
    return False  # Block everything!


root = logging.getLogger()
root.addFilter(block_all)

# Child logger
child = logging.getLogger("myapp")

# Direct log to root: BLOCKED by filter
root.warning("Direct to root")  # Blocked ✓

# Log via child that propagates: NOT BLOCKED!
child.warning("From child")  # Gets through!

# Why? Because propagation goes directly to parent's handlers,
# bypassing the parent's filter() method.
#
# If you want to filter at a central point, add the filter to
# the HANDLER, not the logger.

### Gotcha 3: Logger Level Gates Everything

A common mistake is setting the handler level low but forgetting about the logger level:

In [ ]:
# "Why aren't my DEBUG messages showing?"

logger = logging.getLogger("myapp")
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)  # Handler accepts DEBUG
logger.addHandler(handler)

# But the root logger defaults to WARNING!
# logger.getEffectiveLevel() == 30 (WARNING)

logger.debug("This won't show!")  # Blocked at early level check

# Fix: Set the logger's level too
logger.setLevel(logging.DEBUG)
logger.debug("Now it shows!")  # Works

## Key Takeaways

1. **Two level checks exist**: The logger's effective level (early gate) and each handler's level (per-destination control).

2. **Filters on loggers vs handlers have different scopes**:
   - Logger filters: Apply once, affect all handlers (including via propagation)
   - Handler filters: Apply per-handler, other handlers unaffected

3. **Propagation bypasses parent logger filters**: This is by design but often surprising. If you need centralized filtering, put filters on handlers.

4. **Effective level is inherited, handler level is not**: Loggers look up the hierarchy for their level; handlers just use their own.

5. **NOTSET means different things**: For loggers, it means "inherit." For handlers, it means "accept all."

6. **The flow is**: Log call → Logger level check → Create LogRecord → Logger filters → (For each handler: Handler level check → Handler filters → Emit) → Propagate to parent → Repeat handler loop

## References

- [Python `logging` module source code](https://github.com/python/cpython/blob/main/Lib/logging/__init__.py) - The actual implementation
- [logging HOWTO](https://docs.python.org/3/howto/logging.html) - Official tutorial
- [logging Cookbook](https://docs.python.org/3/howto/logging-cookbook.html) - Common patterns
- [logging module documentation](https://docs.python.org/3/library/logging.html) - API reference